# 🌿 Hierarchical Clustering
**ISLP Chapter 12 · Pattern Recognition for the Rest of Us**

> Hierarchical clustering builds a full tree of clusters — the dendrogram — without requiring you to specify K in advance. You can cut the tree at any height to get any number of clusters and visualize the full clustering structure at once.

### What you'll learn
- Agglomerative vs divisive clustering
- Linkage methods: complete, single, average, Ward
- Reading a dendrogram
- Choosing the number of clusters by cutting the tree
- Comparison with K-means

### Dataset: USArrests + Titanic
### Time: ~50 minutes

## 🎯 Part 1 — Agglomerative Clustering

**Bottom-up (agglomerative):**
1. Start with each point in its own cluster (n clusters)
2. Merge the two *most similar* clusters
3. Repeat until everything is in one cluster

The key question: how do we measure distance *between clusters*?

**Linkage methods:**
- **Complete linkage:** distance = max distance between any two points (tends to produce compact clusters)
- **Single linkage:** distance = min distance (can produce elongated 'chain' clusters)
- **Average linkage:** distance = average of all pairwise distances
- **Ward linkage:** merges clusters that minimize the increase in total within-cluster variance (like K-means objective)

In [ ]:
# Compare linkage methods with dendrograms
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
linkage_methods = ['complete','single','average','ward']

for ax, method in zip(axes.flatten(), linkage_methods):
    Z = linkage(X_s, method=method)
    dendrogram(Z, labels=us_arrests.index.tolist(), ax=ax,
              leaf_font_size=7, color_threshold=0.7*max(Z[:,2]),
              above_threshold_color='#888')
    ax.set_title(f'{method.capitalize()} Linkage', fontsize=12)
    ax.set_ylabel('Distance')
    ax.tick_params(axis='x', rotation=90)

plt.suptitle('Dendrograms: USArrests — 4 Linkage Methods', fontsize=13, y=1.01)
plt.tight_layout(); plt.show()
print('\n📌 Ward linkage (bottom-right) produces the most compact, balanced clusters')
print('   Single linkage (top-right) shows chaining effect — one long chain')

In [ ]:
# Cut the dendrogram to get K clusters
Z_ward = linkage(X_s, method='ward')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, K in zip(axes, [2, 3, 4]):
    labels = fcluster(Z_ward, K, criterion='maxclust') - 1
    dendrogram(Z_ward, labels=us_arrests.index.tolist(), ax=ax,
              leaf_font_size=6, color_threshold=sorted(Z_ward[:,2])[-K+1])
    ax.set_title(f'Cut into K={K} clusters')
    ax.set_ylabel('Distance')
    ax.tick_params(axis='x', rotation=90)
    # Add horizontal cut line
    cut_height = sorted(Z_ward[:,2])[-K+1]
    ax.axhline(cut_height, color='red', lw=2, ls='--', label=f'Cut at {cut_height:.2f}')
    ax.legend(fontsize=8)
plt.suptitle('Cutting the Dendrogram at Different Heights', fontsize=12, y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# Analyze the 3-cluster solution
labels_3 = fcluster(Z_ward, 3, criterion='maxclust') - 1
us_clustered = us_arrests.copy()
us_clustered['Cluster'] = labels_3

print('=== 3-Cluster Solution (Ward Linkage) ===')
for k in range(3):
    states = us_clustered[us_clustered['Cluster']==k].index.tolist()
    print(f'\nCluster {k+1} ({len(states)} states):')
    print(f'  States: {states}')
    print(f'  Means: {us_clustered[us_clustered["Cluster"]==k][us_arrests.columns].mean().round(1).to_dict()}')

print(f'\nSilhouette score (K=3): {silhouette_score(X_s, labels_3):.4f}')

In [ ]:
# Compare Hierarchical vs K-Means
from sklearn.cluster import KMeans
hclust_labels = fcluster(linkage(X_s,'ward'), 3, criterion='maxclust') - 1
kmeans_labels = KMeans(3, n_init=20, random_state=42).fit_predict(X_s)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors_c = ['#1e5fa8','#e85d20','#1a7a45']

for ax, (labels, title) in zip(axes, [
    (hclust_labels, 'Hierarchical (Ward linkage)'),
    (kmeans_labels, 'K-Means (K=3)')]):
    for k in range(3):
        mask = labels==k
        for state in us_arrests.index[mask]:
            idx = list(us_arrests.index).index(state)
            ax.annotate(state, (X_s[idx,0], X_s[idx,2]), fontsize=7, alpha=0.8, color=colors_c[k])
    ax.set_xlabel('Murder (standardized)'); ax.set_ylabel('Rape (standardized)')
    ax.set_title(title)
plt.suptitle('Hierarchical vs K-Means: Murder vs Rape dimensions', fontsize=12, y=1.02)
plt.tight_layout(); plt.show()

from sklearn.metrics import adjusted_rand_score
print(f'\nAdjusted Rand Index (agreement): {adjusted_rand_score(hclust_labels, kmeans_labels):.4f}')
print('(1.0 = identical, 0.0 = random agreement)')

In [ ]:
answers = {
    'q1': '',  # What is the main advantage of hierarchical clustering over K-means?
    'q2': '',  # What does the height of a merge in the dendrogram represent?
    'q3': '',  # Which linkage method minimizes within-cluster variance (most like K-means)?
    'q4': '',  # What is the 'chaining' effect and which linkage method causes it?
    'q5': '',  # When would you prefer hierarchical clustering over K-means?
}
missing=[k for k,v in answers.items() if not str(v).strip()]
print('❌ Still empty: '+str(missing) if missing else '✅ Done! File → Save a copy in GitHub')

## 📚 Further Reading
- [ISLP Ch 12.4](https://intro-stat-learning.github.io/ISLP/) — Hierarchical clustering
- [scipy.cluster.hierarchy docs](https://docs.scipy.org/doc/scipy/reference/cluster.hierarchy.html)
- [K-Means notebook](kmeans.ipynb) — flat clustering comparison

---
*Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*